# 02 Bronze - Retail

**Audience:** participants learning the AIDP medallion pattern with PySpark.

**Prerequisites:** use the lab's shared compute and run the previous notebook first.

**Learning goal:** Preserves source values and lineage while converting each dataset to Delta.

## Outline

1. Inspect the participant-scoped inputs.
2. Transform and persist this medallion layer.
3. Register external tables when this layer owns them.
4. Verify the row counts printed by the final statements.


In [ ]:
import re
# oidlUtils is injected by AIDP Workbench; no import is required.

def required_parameter(name):
    value = oidlUtils.parameters.getParameter(name, "")
    if value is None or not str(value).strip():
        raise ValueError(f"Missing AIDP job parameter: {name}")
    return str(value).strip()

participant_key = required_parameter("participant_key")
lab_id = required_parameter("lab_id")
workspace_root = required_parameter("workspace_root")
bucket_name = required_parameter("bucket_name")
objectstorage_namespace = required_parameter("objectstorage_namespace")
catalog_name = required_parameter("catalog_name")

participant_match = re.fullmatch(r"u([1-9][0-9]*)", participant_key)
if participant_match is None or int(participant_match.group(1)) < 101:
    raise ValueError("Invalid participant_key")
if lab_id != 'retail':
    raise ValueError("This notebook belongs to a different lab")
if not workspace_root.startswith("/Workspace/medallon/"):
    raise ValueError("Invalid workspace_root")
if catalog_name != f"{participant_key}_aidp":
    raise ValueError("Invalid participant catalog")
spark.conf.set("spark.aidp.lineage.enabled", "true")

def table(layer, logical_name):
    prefix = f"{lab_id}_"
    physical_name = logical_name if logical_name.startswith(prefix) else prefix + logical_name
    return f"{catalog_name}.oci_{layer}.{participant_key}_{physical_name}"

from pyspark.sql import functions as F

specs = {'customers': {'filename': 'customers.csv', 'primary_key': ['customer_id'], 'foreign_keys': [], 'columns': [{'name': 'participant_key', 'type': 'STRING', 'required': True}, {'name': 'source_row_id', 'type': 'STRING', 'required': True}, {'name': 'customer_id', 'type': 'STRING', 'required': True}, {'name': 'segment', 'type': 'STRING', 'required': True}, {'name': 'region', 'type': 'STRING', 'required': True}, {'name': 'loyalty_tier', 'type': 'STRING', 'required': True}, {'name': 'signup_date', 'type': 'DATE', 'required': True}, {'name': 'status', 'type': 'STRING', 'required': True}, {'name': 'updated_at', 'type': 'TIMESTAMP', 'required': True}]}, 'products': {'filename': 'products.csv', 'primary_key': ['product_id'], 'foreign_keys': [], 'columns': [{'name': 'participant_key', 'type': 'STRING', 'required': True}, {'name': 'source_row_id', 'type': 'STRING', 'required': True}, {'name': 'product_id', 'type': 'STRING', 'required': True}, {'name': 'category', 'type': 'STRING', 'required': True}, {'name': 'brand_label', 'type': 'STRING', 'required': True}, {'name': 'unit_cost', 'type': 'DOUBLE', 'required': True}, {'name': 'list_price', 'type': 'DOUBLE', 'required': True}, {'name': 'status', 'type': 'STRING', 'required': True}, {'name': 'updated_at', 'type': 'TIMESTAMP', 'required': True}]}, 'orders': {'filename': 'orders.csv', 'primary_key': ['order_id'], 'foreign_keys': [['customer_id', 'customers', 'customer_id']], 'columns': [{'name': 'participant_key', 'type': 'STRING', 'required': True}, {'name': 'source_row_id', 'type': 'STRING', 'required': True}, {'name': 'order_id', 'type': 'STRING', 'required': True}, {'name': 'customer_id', 'type': 'STRING', 'required': True}, {'name': 'order_time', 'type': 'TIMESTAMP', 'required': True}, {'name': 'channel', 'type': 'STRING', 'required': True}, {'name': 'region', 'type': 'STRING', 'required': True}, {'name': 'currency', 'type': 'STRING', 'required': True}, {'name': 'order_status', 'type': 'STRING', 'required': True}, {'name': 'discount_amount', 'type': 'DOUBLE', 'required': True}, {'name': 'declared_total', 'type': 'DOUBLE', 'required': True}, {'name': 'updated_at', 'type': 'TIMESTAMP', 'required': True}]}, 'order_items': {'filename': 'order_items.csv', 'primary_key': ['order_id', 'line_number'], 'foreign_keys': [['order_id', 'orders', 'order_id'], ['product_id', 'products', 'product_id']], 'columns': [{'name': 'participant_key', 'type': 'STRING', 'required': True}, {'name': 'source_row_id', 'type': 'STRING', 'required': True}, {'name': 'order_id', 'type': 'STRING', 'required': True}, {'name': 'line_number', 'type': 'BIGINT', 'required': True}, {'name': 'product_id', 'type': 'STRING', 'required': True}, {'name': 'quantity', 'type': 'BIGINT', 'required': True}, {'name': 'unit_price', 'type': 'DOUBLE', 'required': True}, {'name': 'discount_amount', 'type': 'DOUBLE', 'required': True}, {'name': 'updated_at', 'type': 'TIMESTAMP', 'required': True}]}}
sources = {"customers": f"oci://{bucket_name}@{objectstorage_namespace}/01_landing/users/{participant_key}/retail/customers/", "order_items": f"oci://{bucket_name}@{objectstorage_namespace}/01_landing/users/{participant_key}/retail/order_items/", "orders": f"oci://{bucket_name}@{objectstorage_namespace}/01_landing/users/{participant_key}/retail/orders/", "products": f"oci://{bucket_name}@{objectstorage_namespace}/01_landing/users/{participant_key}/retail/products/"}
destinations = {"customers": f"oci://{bucket_name}@{objectstorage_namespace}/02_bronze/users/{participant_key}/retail/customers/", "order_items": f"oci://{bucket_name}@{objectstorage_namespace}/02_bronze/users/{participant_key}/retail/order_items/", "orders": f"oci://{bucket_name}@{objectstorage_namespace}/02_bronze/users/{participant_key}/retail/orders/", "products": f"oci://{bucket_name}@{objectstorage_namespace}/02_bronze/users/{participant_key}/retail/products/"}
landing_tables = {"customers": f"{participant_key}_retail_customers", "order_items": f"{participant_key}_retail_order_items", "orders": f"{participant_key}_retail_orders", "products": f"{participant_key}_retail_products"}

for dataset, spec in specs.items():
    frame = (spark.table(table("landing", dataset))
        .withColumn("_source_file", F.input_file_name())
        .withColumn("_ingested_at", F.current_timestamp()))
    landing_count = frame.count()
    frame.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(table("bronze", dataset))
    bronze_count = spark.table(table("bronze", dataset)).count()
    assert bronze_count == landing_count, f"Bronze count mismatch for {dataset}"
    print(f"Bronze {dataset}: {bronze_count} rows")


## Expected result

Four Landing CSV tables and four Bronze Delta tables are registered.

**Exercise:** rerun this notebook and confirm that counts do not increase. All writes use
participant-exclusive paths and overwrite mode, so a second run is idempotent.

**Common pitfall:** do not replace the participant paths with shared locations. That would mix
different students' data. As an extension, query the registered tables with `spark.sql`.
